# Lane Seven — Demand Forecasting
# **v7.5 — Backtest-Based Bias Calibration** 🎯

> **v7.4.1 showed BiasRatio ≈ 1.106 — systematic over-forecasting.**
> v7.5 fixes this with a true rolling backtest calibration layer.

## What v7.5 adds over v7.4

| Layer | v7.4 | v7.5 |
|-------|------|------|
| Upstream model | StyleCode XGBoost/LightGBM | **same** |
| Calibration data | CV fold predictions (often missing) | **Rolling backtest (always generated)** |
| Evidence quality | Often WEAK or NONE | **STRONG or MODERATE for most StyleCodes** |
| Calibration path | Fallback seasonal naive | **True out-of-sample backtest predictions** |
| Allocation | recency_only | **same (v7.2 champion)** |

## No-leakage guarantee

- Backtest origins are strictly before Jan 2026
- ForecastMonths used for calibration are ≤ 2025-12
- Jan–Feb 2026 actuals are used ONLY for holdout evaluation

## Required inputs

```
data/gold_fact_monthly_demand_v2.csv
data/dim_date.csv
data/dim_product.csv
outputs/v7_4_best_models.csv  ← or any prior v7.x best models CSV
```

## Output files

```
outputs/v7_5_stylecode_backtest_predictions.csv
outputs/v7_5_stylecode_calibration_table.csv
outputs/v7_5_stylecode_forecasts_raw.csv
outputs/v7_5_stylecode_forecasts_calibrated.csv
outputs/v7_5_production_sku_forecasts.csv    ← MAIN DELIVERABLE
outputs/v7_5_holdout_evaluation.csv
outputs/v7_5_error_decomposition.csv
outputs/v7_5_bias_analysis.csv
outputs/v7_5_vs_prior_versions.csv
outputs/v7_5_validation_report.csv
```

## Section 0 — Environment setup

In [3]:
import subprocess, sys
packages = ['pandas','numpy','scikit-learn','xgboost','lightgbm','statsmodels','matplotlib','openpyxl']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+packages)
print('All packages installed.')

All packages installed.


In [4]:
import sys, logging
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
DATA_DIR     = NOTEBOOK_DIR / 'data'
OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S', handlers=[logging.StreamHandler(sys.stdout)], force=True,
)
for _noisy in ['lightgbm','xgboost','prophet','cmdstanpy']:
    logging.getLogger(_noisy).setLevel(logging.ERROR)

GOLD_OPTIONS = ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
gold_present = [f for f in GOLD_OPTIONS if (DATA_DIR/f).exists()]
date_ok = (DATA_DIR/'dim_date.csv').exists()
prod_ok = (DATA_DIR/'dim_product.csv').exists()

print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'  {chr(10003) if gold_present else chr(10007)}  {gold_present[0] if gold_present else "MISSING"}')
print(f'  {chr(10003) if date_ok else chr(10007)+" MISSING"}  dim_date.csv')
print(f'  {chr(10003) if prod_ok else chr(10007)+" MISSING"}  dim_product.csv')
if not (gold_present and date_ok and prod_ok):
    raise FileNotFoundError('Required input files missing.')
print('\nSetup complete.')

DATA_DIR   : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.5\data
OUTPUT_DIR : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.5\outputs
  ✓  gold_fact_monthly_demand_v2.csv
  ✓  dim_date.csv
  ✓  dim_product.csv

Setup complete.


## Section 1 — Configuration

In [5]:
from lane7_forecast.forecast_adjustments import get_config

PHASE = 1
# Dynamically detect holdout months from dim_date
_dim_date_check = pd.read_csv(DATA_DIR / "dim_date.csv", parse_dates=["MonthStart"])
_dim_date_check["MonthStart"] = (
    _dim_date_check["MonthStart"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

_candidate_split_cols = [
    "Split", "TrainHoldout", "DataSplit",
    "DatasetSplit", "Set", "IsTrain"
]

_holdout_mask = None
_holdout_source_col = None

for _col in _candidate_split_cols:
    if _col not in _dim_date_check.columns:
        continue

    if _col == "IsTrain":
        _vals = _dim_date_check[_col]
        if _vals.dtype == bool:
            _mask = (_vals == False)
        else:
            _mask = _vals.astype(str).str.strip().str.lower().isin(["0", "false", "holdout"])
    else:
        _mask = _dim_date_check[_col].astype(str).str.strip().str.upper().eq("HOLDOUT")

    if _mask.any():
        _holdout_mask = _mask
        _holdout_source_col = _col
        break

if _holdout_mask is None:
    raise ValueError("Could not detect holdout months from dim_date.csv.")

HOLDOUT_MONTHS = sorted(_dim_date_check.loc[_holdout_mask, "MonthStart"].unique())
HOLDOUT_MONTHS = [pd.Timestamp(m) for m in HOLDOUT_MONTHS]

if len(HOLDOUT_MONTHS) == 0:
    raise ValueError("No holdout months detected.")

HOLDOUT_START = HOLDOUT_MONTHS[0].strftime("%Y-%m-%d")

BACKTEST_END = HOLDOUT_MONTHS[0] - pd.DateOffset(months=1)
BACKTEST_END = pd.Timestamp(BACKTEST_END.year, BACKTEST_END.month, 1)

N_HOLDOUT_MONTHS = len(HOLDOUT_MONTHS)

print(f"Detected holdout source column: {_holdout_source_col}")
print("Detected holdout months:")
for _m in HOLDOUT_MONTHS:
    print(" ", _m.strftime("%Y-%m"))



ADJ_CONFIG = get_config(
    shrink_threshold=1.3, shrink_factor=0.75, recursive_alpha=0.6,
    blend_threshold=0.20, blend_weight=0.7, intermittent_cap_multiplier=2.0,
)

ALLOC_PARAMS = dict(lookback_months=12, min_lookback_months=6, w_recent=3, w_mid=2, w_old=1)

# Backtest configuration
BACKTEST_CONFIG = dict(
    origin_step_months=3,   # quarterly origins
    n_origins=8,            # 8 origins = 2 years back
)

# Calibration evidence thresholds (v7.5 specific)
CALIB_CONFIG = dict(
    strong_n=6, strong_units=500,
    moderate_n=3, moderate_units=200,
    min_factor=0.85, max_factor=1.15,
)

print(f'Holdout months  : {[m.strftime("%Y-%m") for m in HOLDOUT_MONTHS]}')
print(f'Backtest end    : {BACKTEST_END.strftime("%Y-%m")}')
print(f'Backtest origins: {BACKTEST_CONFIG["n_origins"]} × every {BACKTEST_CONFIG["origin_step_months"]} months')
print(f'Calib range     : [{CALIB_CONFIG["min_factor"]}, {CALIB_CONFIG["max_factor"]}]')

Detected holdout source column: Split
Detected holdout months:
  2026-01
  2026-02
  2026-03
  2026-04
Holdout months  : ['2026-01', '2026-02', '2026-03', '2026-04']
Backtest end    : 2025-12
Backtest origins: 8 × every 3 months
Calib range     : [0.85, 1.15]


## Section 2 — Load Data and Build StyleCode Panel

In [6]:
from lane7_forecast.pipeline import run_v7_prep, run_cv

scode_prep = run_v7_prep(
    data_dir=DATA_DIR, phase=PHASE,
    lookback_months=ALLOC_PARAMS['lookback_months'],
    min_lookback_months=ALLOC_PARAMS['min_lookback_months'],
)

tables      = scode_prep['tables']
gold_df     = tables['demand']
dim_prod_df = scode_prep['dim_product']
dim_date_df = tables['dim_date']

scode_segments = scode_prep['segments']
scode_segments.to_csv(OUTPUT_DIR/'v7_5_stylecode_segments.csv', index=False)

print(f'StyleCode panel: {scode_prep["panel_seg"]["SKU"].nunique():,} StyleCodes')
for seg, n in scode_segments['Segment'].value_counts().items():
    print(f'  {seg:<15} {n:5d}')

14:48:37  INFO      === v7 Step 1: StyleCode-level data prep (phase=1) ===
14:48:37  INFO      Loaded gold_fact_monthly_demand: 119498 rows, 4 cols
14:48:37  INFO      Loaded dim_date: 116 rows, 6 cols
14:48:37  INFO      Loaded dim_product: 3806 rows, 15 cols
14:48:37  WARNING   Optional file not found, skipping: C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.5\data\dim_customer.csv
14:48:37  INFO      Training period end (phase 1): 2025-12
14:48:38  INFO      [v7] STANDALONE (StyleCode level): 468 SKUs, 538 rows, 19,551 units (0.03% of total)
14:48:38  INFO      [v7] StyleCode demand table: 2336 rows, 61 unique StyleCodes, 2017-05 → 2026-04
14:48:38  INFO      Training period end (phase 1): 2025-12
14:48:38  INFO      Rows after date filter (<= 2025-12): 2190
14:48:38  WARNING   Attribute columns missing from demand and no dim_product provided: ['Category', 'StyleCode', 'ColorCode', 'SizeCode', 'StyleColor']. Filling with 'UNKNOWN'.
14:48:38  INFO   

In [7]:
USE_EXISTING_CV = False
_cv_path   = OUTPUT_DIR / 'v7_5_cv_results.csv'
_best_path = OUTPUT_DIR / 'v7_5_best_models.csv'

# Try to load from v7.4 if v7.5 not yet run
_v74_best_path = OUTPUT_DIR / 'v7_4_best_models.csv'

v75_bm = {}

if USE_EXISTING_CV and _best_path.exists():
    print('Loading cached v7.5 CV results...')
    best_models_v75 = pd.read_csv(_best_path)
    for h in [1,3]:
        v75_bm[h] = best_models_v75[best_models_v75['HorizonMonths']==h].copy()
elif _v74_best_path.exists():
    print(f'Reusing v7.4 best models from {_v74_best_path.name}')
    best_models_v75 = pd.read_csv(_v74_best_path)
    for h in [1,3]:
        v75_bm[h] = best_models_v75[best_models_v75['HorizonMonths']==h].copy()
else:
    print('Running StyleCode CV...')
    for h, cfg in {1:dict(n_folds=6,step_months=3,min_train_months=24),
                   3:dict(n_folds=5,step_months=3,min_train_months=24)}.items():
        print(f'\n--- H={h} ---')
        cv_df, best_df = run_cv(scode_prep, horizon_months=h, **cfg)
        v75_bm[h] = best_df

best_models_v75 = pd.concat([df for df in v75_bm.values() if not df.empty], ignore_index=True)
best_models_v75.to_csv(OUTPUT_DIR/'v7_5_best_models.csv', index=False)

print('\n=== v7.5 Best Models ===')
print(best_models_v75[['Segment','HorizonMonths','BestModel','mean_WMAPE']].round(2).to_string(index=False))

Reusing v7.4 best models from v7_4_best_models.csv

=== v7.5 Best Models ===
     Segment  HorizonMonths    BestModel  mean_WMAPE
     REGULAR              1     LightGBM       28.29
INTERMITTENT              1   CrostonSBA       86.16
        DEAD              1 ZeroForecast         NaN
     REGULAR              3      XGBoost       28.70
INTERMITTENT              3   CrostonSBA       87.44
        DEAD              3 ZeroForecast         NaN


## Section 3 — Run Rolling StyleCode Backtest

> **Leakage guard:** all origin months are strictly before Jan 2026.
> Actuals are only joined for ForecastMonths ≤ 2025-12.
> This section may take 5–20 minutes depending on StyleCode count.

In [8]:
from lane7_forecast.backtest_calibration_v75 import run_stylecode_backtest

USE_EXISTING_BACKTEST = False
_bt_path = OUTPUT_DIR / 'v7_5_stylecode_backtest_predictions.csv'

if USE_EXISTING_BACKTEST and _bt_path.exists():
    print('Loading cached backtest predictions...')
    backtest_df = pd.read_csv(_bt_path)
    print(f'Loaded {len(backtest_df):,} rows from {_bt_path.name}')
else:
    print('Running rolling StyleCode backtest...')
    print(f'  Backtest end   : {BACKTEST_END.strftime("%Y-%m")}')
    print(f'  Origins        : {BACKTEST_CONFIG["n_origins"]} × every '
          f'{BACKTEST_CONFIG["origin_step_months"]} months')
    print(f'  Horizons       : H=1 and H=3')
    print()

    backtest_df = run_stylecode_backtest(
        gold_df=gold_df,
        dim_product_df=dim_prod_df,
        dim_date_df=dim_date_df,
        best_models_df=best_models_v75,
        adjustment_config=ADJ_CONFIG,
        backtest_end=BACKTEST_END,
        horizon_months_list=[1, 3],
        **BACKTEST_CONFIG,
        phase=PHASE,
    )
    backtest_df.to_csv(OUTPUT_DIR/'v7_5_stylecode_backtest_predictions.csv', index=False)

print(f'\nBacktest predictions: {len(backtest_df):,} rows')
print(f'StyleCodes covered  : {backtest_df["StyleCodeDesc"].nunique():,}')
print(f'Horizon distribution:')
print(backtest_df['HorizonMonths'].value_counts().sort_index().to_string())
print()
print('Overall backtest performance (all origins, all months):')
for h in sorted(backtest_df['HorizonMonths'].unique()):
    h_df = backtest_df[(backtest_df['HorizonMonths']==h) & (backtest_df['ActualUnits']>0)]
    if not h_df.empty:
        wmape = h_df['AbsError'].sum() / h_df['ActualUnits'].sum() * 100
        bias  = h_df['PredictedUnits'].sum() / h_df['ActualUnits'].sum()
        print(f'  H={h}: WMAPE={wmape:.1f}%  BiasRatio={bias:.4f}')

Running rolling StyleCode backtest...
  Backtest end   : 2025-12
  Origins        : 8 × every 3 months
  Horizons       : H=1 and H=3

14:48:52  INFO      [v7.5 backtest] 8 origins × 2 horizons  (backtest_end=2025-12)
14:48:52  INFO      [v7.5 backtest] Origin 2023-12
14:48:52  INFO      Training period end (phase 1): 2023-11
14:48:52  INFO      Rows after date filter (<= 2023-11): 1524
14:48:52  WARNING   Attribute columns missing from demand and no dim_product provided: ['Category', 'StyleCode', 'ColorCode', 'SizeCode', 'StyleColor']. Filling with 'UNKNOWN'.
14:48:52  INFO      Panel built: 2671 rows, 43 unique SKUs, date range 2017-05 to 2023-11
14:48:52  INFO      SKU segmentation complete — REGULAR: 22 | INTERMITTENT: 1 | DEAD: 20
14:48:52  INFO      === Step 3: Forecast generation (horizon=1, phase=1) ===
14:48:52  INFO      Horizon 1-month: dropped 129 rows with null required lags (['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3'])
14:48:52  INFO      Features created for horizon=1 

## Section 4 — Build Calibration Table

In [9]:
from lane7_forecast.backtest_calibration_v75 import (
    build_v75_calibration_table, validate_calibration_table,
)

calib_table = build_v75_calibration_table(
    backtest_df=backtest_df,
    backtest_end=BACKTEST_END,
    horizon_months_list=[1, 3],
    **CALIB_CONFIG,
)
calib_table.to_csv(OUTPUT_DIR/'v7_5_stylecode_calibration_table.csv', index=False)

val_calib = validate_calibration_table(
    calibration_df=calib_table,
    min_factor=CALIB_CONFIG['min_factor'],
    max_factor=CALIB_CONFIG['max_factor'],
    backtest_end=BACKTEST_END,
    backtest_df=backtest_df,
)

print('=== v7.5 Calibration Table Summary ===')
print(f'  Rows             : {val_calib["n_rows"]}')
print(f'  Applied          : {val_calib["n_applied"]}')
print(f'  By tier          : {val_calib["n_by_tier"]}')
print(f'  Factors in range : {val_calib["factors_in_range"]}')
print(f'  Duplicate keys   : {val_calib["has_duplicate_keys"]}')
print(f'  Leakage clean    : {val_calib["leakage_check_passed"]}')
if val_calib['warnings']:
    print(f'  Warnings         : {val_calib["warnings"]}')
print()
if not calib_table.empty:
    applied = calib_table[calib_table['calibration_applied']]
    print(f'Factors applied: {len(applied)} (StyleCode, HorizonMonths) pairs')
    if not applied.empty:
        print()
        print('Distribution of applied calibration factors:')
        print(f'  Mean   : {applied["calibration_factor"].mean():.4f}')
        print(f'  Std    : {applied["calibration_factor"].std():.4f}')
        print(f'  Min    : {applied["calibration_factor"].min():.4f}')
        print(f'  Max    : {applied["calibration_factor"].max():.4f}')
        print()
        print('Sample applied factors (sorted by factor ascending):')
        print(applied.nsmallest(10,'calibration_factor')[
            ['StyleCodeDesc','HorizonMonths','n_backtest_observations',
             'raw_bias_ratio','calibration_factor','evidence_tier']
        ].to_string(index=False))

14:50:32  INFO      [v7.5 calib] Table: 96 rows | STRONG=52 MODERATE=4 WEAK=0 NONE=40 | applied=56
=== v7.5 Calibration Table Summary ===
  Rows             : 96
  Applied          : 56
  By tier          : {'STRONG': 52, 'NONE': 40, 'MODERATE': 4}
  Factors in range : True
  Duplicate keys   : False
  Leakage clean    : True

Factors applied: 56 (StyleCode, HorizonMonths) pairs

Distribution of applied calibration factors:
  Mean   : 1.0764
  Std    : 0.0987
  Min    : 0.8500
  Max    : 1.1500

Sample applied factors (sorted by factor ascending):
                          StyleCodeDesc  HorizonMonths  n_backtest_observations  raw_bias_ratio  calibration_factor evidence_tier
   LS13004 French Terry Raglan Crewneck              1                        8          1.3216              0.8500        STRONG
    LS15009 Heavyweight Long Sleeve Tee              1                        8          1.4493              0.8500        STRONG
                    LS12000 Crop Hoodie              1  

## Section 5 — Generate Raw StyleCode Forecasts

In [10]:
from lane7_forecast.pipeline import run_forecasts
from lane7_forecast.data_prep import _resolve_training_end, build_panel
from lane7_forecast.segmentation import segment_skus, attach_segment

train_end       = _resolve_training_end(dim_date_df, phase=1)
standalone_skus = scode_prep.get('stylecode_standalone_skus', [])

v75_raw_fc  = {}
v75_sa_fc   = {}

for horizon in [3, 1]:
    bm_h = best_models_v75[best_models_v75['HorizonMonths']==horizon]
    if bm_h.empty:
        print(f'H={horizon}: no best model — skipping.')
        continue
    n_pred = N_HOLDOUT_MONTHS

    raw_fc = run_forecasts(
        prep=scode_prep, best_models_df=bm_h, horizon_months=horizon,
        forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
        phase=PHASE, model_version=f'v7.5-raw-H{horizon}',
        output_path=None, append=False, adjustment_config=ADJ_CONFIG,
    )
    v75_raw_fc[horizon] = raw_fc
    raw_fc.to_csv(OUTPUT_DIR/f'v7_5_stylecode_forecasts_raw_H{horizon}.csv', index=False)
    print(f'H={horizon} raw: {len(raw_fc):,} rows, {raw_fc["Key"].nunique():,} StyleCodes')

    # STANDALONE pass-through
    if standalone_skus:
        sa_gold = gold_df[gold_df['SKU'].isin(standalone_skus)].copy()
        if not sa_gold.empty:
            sa_p = build_panel(sa_gold, dim_date_df, phase=PHASE, dim_product_df=dim_prod_df)
            sa_s = segment_skus(sa_p)
            sa_prep = {'tables':tables,'panel':sa_p,'segments':sa_s,
                       'panel_seg':attach_segment(sa_p,sa_s)}
            v75_sa_fc[horizon] = run_forecasts(
                prep=sa_prep, best_models_df=bm_h, horizon_months=horizon,
                forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
                phase=PHASE, model_version=f'v7.5-standalone-H{horizon}',
                output_path=None, append=False, adjustment_config=ADJ_CONFIG,
            )

# Combine H=3 and H=1 raw for single file
raw_combined = pd.concat([df for df in v75_raw_fc.values() if not df.empty], ignore_index=True)
raw_combined.to_csv(OUTPUT_DIR/'v7_5_stylecode_forecasts_raw.csv', index=False)
print(f'\nSaved v7_5_stylecode_forecasts_raw.csv ({len(raw_combined):,} rows)')

14:50:32  INFO      Training period end (phase 1): 2025-12
14:50:32  INFO      === Step 3: Forecast generation (horizon=3, phase=1) ===
14:50:32  INFO      Horizon 3-month: dropped 590 rows with null required lags (['lag_1', 'lag_3', 'lag_6', 'lag_12', 'rolling_mean_6'])
14:50:32  INFO      Features created for horizon=3 — panel shape: (3260, 34)
14:50:35  INFO      Trained XGBoost on 1485 rows, 19 features
14:50:35  INFO      Retrained XGBoost for segment=REGULAR on 1485 rows
14:50:35  INFO      Generating 3-month forecasts for 52 SKUs, 4 months from 2026-01 (adjustments=enabled)
14:50:39  INFO      Forecast complete: 208 rows, horizon=3
H=3 raw: 208 rows, 52 StyleCodes
14:50:39  INFO      Training period end (phase 1): 2025-12
14:50:39  INFO      Rows after date filter (<= 2025-12): 537
14:50:41  INFO      dim_product joined onto panel. Unknown counts per attr: {'StyleCode': 26877}
14:50:41  INFO      Panel built: 26961 rows, 468 unique SKUs, date range 2017-12 to 2025-12
14:50:41  I

## Section 6 — Apply v7.5 Calibration

In [11]:
from lane7_forecast.backtest_calibration_v75 import apply_v75_calibration

v75_cal_fc = {}

for horizon in [3, 1]:
    if horizon not in v75_raw_fc:
        continue

    raw_fc = v75_raw_fc[horizon]
    calib_h = calib_table[calib_table['HorizonMonths']==horizon]

    cal_fc = apply_v75_calibration(
        scode_forecasts_df=raw_fc,
        calibration_df=calib_h,
    )
    v75_cal_fc[horizon] = cal_fc

    n_adj = int(cal_fc['CalibrationApplied'].sum()) if 'CalibrationApplied' in cal_fc.columns else 0
    print(f'H={horizon}: calibration applied to {n_adj}/{len(cal_fc)} rows')

cal_combined = pd.concat([df for df in v75_cal_fc.values() if not df.empty], ignore_index=True)
cal_combined.to_csv(OUTPUT_DIR/'v7_5_stylecode_forecasts_calibrated.csv', index=False)
print(f'\nSaved v7_5_stylecode_forecasts_calibrated.csv ({len(cal_combined):,} rows)')

# Quick bias check: raw vs calibrated at StyleCode aggregate
print()
print('=== Aggregate StyleCode forecast: raw vs calibrated ===')
for horizon in [3, 1]:
    if horizon not in v75_raw_fc or horizon not in v75_cal_fc:
        continue
    raw_tot = v75_raw_fc[horizon]['ForecastUnits'].sum()
    cal_tot = v75_cal_fc[horizon]['ForecastUnits'].sum()
    change  = (cal_tot - raw_tot) / max(1, raw_tot) * 100
    print(f'  H={horizon}: raw={raw_tot:,.0f}  calibrated={cal_tot:,.0f}  '
          f'change={change:+.1f}%')

14:51:00  INFO      [v7.5 calib] Applied factors: 112 / 208 rows adjusted
H=3: calibration applied to 112/208 rows
14:51:00  INFO      [v7.5 calib] Applied factors: 112 / 208 rows adjusted
H=1: calibration applied to 112/208 rows

Saved v7_5_stylecode_forecasts_calibrated.csv (416 rows)

=== Aggregate StyleCode forecast: raw vs calibrated ===
  H=3: raw=4,086,853  calibrated=4,546,991  change=+11.3%
  H=1: raw=4,482,249  calibrated=4,752,534  change=+6.0%


## Section 7 — Production Allocation: v7.2 recency_only

In [12]:
from lane7_forecast.allocation_v72 import ALLOCATION_VARIANTS, run_allocation_variant

v75_scol_fc = {}
v75_sku_fc  = {}

for horizon in [3, 1]:
    if horizon not in v75_cal_fc:
        continue

    sa_fc = v75_sa_fc.get(horizon, None)

    alloc = run_allocation_variant(
        variant_name='recency_only',
        variant_cfg=ALLOCATION_VARIANTS['recency_only'],
        scode_forecasts_df=v75_cal_fc[horizon],
        gold_df=gold_df,
        dim_product_df=dim_prod_df,
        train_end=train_end,
        standalone_fc_df=sa_fc,
        **ALLOC_PARAMS,
        smooth_alpha=0.0,
        cap_rel_increase=999.0,
    )
    v75_scol_fc[horizon] = alloc['stylecolor_forecasts']
    v75_sku_fc[horizon]  = alloc['sku_forecasts']

    val = alloc['validation']
    print(f'H={horizon}: sc→scol={val["sc_to_scol_sum_ok"]}  '
          f'scol→sku={val["scol_to_sku_sum_ok"]}  '
          f'no_neg={val["no_negatives"]}  '
          f'SKUs={alloc["sku_forecasts"]["Key"].nunique():,}')

14:51:01  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
14:51:16  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0008)  scol→sku=True (diff=0.0003)  no_neg=True
H=3: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882
14:51:16  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
14:51:31  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0005)  scol→sku=True (diff=0.0003)  no_neg=True
H=1: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882


## Section 8 — Build Production SKU Table

In [13]:
from lane7_forecast.production_outputs_v75 import build_v75_production_sku_table

all_sku = pd.concat([df for df in v75_sku_fc.values() if not df.empty], ignore_index=True)

prod_sku = build_v75_production_sku_table(
    sku_fc_df=all_sku,
    dim_product_df=dim_prod_df,
    calibration_df=calib_table,
    allocation_method='recency_only_v7.2',
)
prod_sku.to_csv(OUTPUT_DIR/'v7_5_production_sku_forecasts.csv', index=False)

print('=== v7.5 Production SKU Forecast Table ===')
print(f'  Rows          : {len(prod_sku):,}')
print(f'  Unique SKUs   : {prod_sku["SKU"].nunique():,}')
print(f'  MonthStart    : {sorted(prod_sku["MonthStart"].unique())}')
print(f'  Horizons      : {sorted(prod_sku["HorizonMonths"].unique())}')
print()
print('Sample (H=3, first 5 rows):')
print(prod_sku[prod_sku['HorizonMonths']==3].head(5)[[
    'MonthStart','HorizonMonths','SKU','StyleCodeDesc',
    'ForecastUnits','CalibrationApplied','CalibrationFactor','AllocationMethod'
]].to_string(index=False))

14:51:40  INFO      [v7.5] Production SKU table: 31056 rows, 3882 SKUs
=== v7.5 Production SKU Forecast Table ===
  Rows          : 31,056
  Unique SKUs   : 3,882
  MonthStart    : [Timestamp('2026-01-01 00:00:00'), Timestamp('2026-02-01 00:00:00'), Timestamp('2026-03-01 00:00:00'), Timestamp('2026-04-01 00:00:00')]
  Horizons      : [np.int64(1), np.int64(3)]

Sample (H=3, first 5 rows):
MonthStart  HorizonMonths             SKU StyleCodeDesc  ForecastUnits  CalibrationApplied  CalibrationFactor  AllocationMethod
2026-01-01              3     002 Vintage           NaN            0.0               False                1.0 recency_only_v7.2
2026-01-01              3 002 Vintage Tee           NaN            0.0               False                1.0 recency_only_v7.2
2026-01-01              3    004 V-Hoodie           NaN            0.0               False                1.0 recency_only_v7.2
2026-01-01              3    12000 Crop H           NaN            0.0               False      

## Section 9 — Validation Report

In [14]:
from lane7_forecast.production_outputs_v75 import build_v75_validation_report

if 3 in v75_cal_fc and 3 in v75_scol_fc and 3 in v75_sku_fc:
    val_report = build_v75_validation_report(
        scode_fc=v75_cal_fc[3],
        scol_fc=v75_scol_fc[3],
        sku_fc=v75_sku_fc[3],
        calibration_df=calib_table,
        backtest_df=backtest_df,
        backtest_end=BACKTEST_END,
        min_factor=CALIB_CONFIG['min_factor'],
        max_factor=CALIB_CONFIG['max_factor'],
    )
    val_report.to_csv(OUTPUT_DIR/'v7_5_validation_report.csv', index=False)

    print('=== v7.5 Validation Report ===')
    print(val_report.to_string(index=False))
    n_fail = (~val_report['passed']).sum()
    print()
    print('✅ ALL CHECKS PASSED' if n_fail == 0 else f'❌ {n_fail} check(s) FAILED')

14:51:40  INFO      [v7.5] Validation: 7 checks, 7 passed, 0 failed
=== v7.5 Validation Report ===
                                                    check  passed                                      detail
                    StyleCode totals == StyleColor totals    True                             max_diff=0.0008
                          StyleColor totals == SKU totals    True                             max_diff=0.0003
                                No negative ForecastUnits    True                             0 negative rows
       No duplicate (SKU, MonthStart, HorizonMonths) rows    True                                0 duplicates
                 Calibration factors within allowed range    True min=0.8500, max=1.1500, allowed=[0.85,1.15]
Calibration table has no duplicate StyleCode-Horizon keys    True                                0 duplicates
  No leakage check passed (ForecastMonths ≤ backtest_end)    True    0 rows with ForecastMonth > backtest_end

✅ ALL CHECKS PASSED


## Section 10 — Holdout Evaluation (Jan–Feb 2026)

In [15]:
from lane7_forecast.production_outputs_v75 import (
    score_v75_holdout, build_v75_error_decomposition,
)

_gpath = next((DATA_DIR/f for f in ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
               if (DATA_DIR/f).exists()), None)
actuals_df = pd.read_csv(_gpath, parse_dates=['MonthStart'])
actuals_df['MonthStart'] = actuals_df['MonthStart'].dt.to_period('M').dt.to_timestamp()
actuals_df['SKU'] = actuals_df['SKU'].astype(str).str.strip()

v75_eval, v75_preds = score_v75_holdout(
    sku_fc=prod_sku, actuals_df=actuals_df, holdout_months=HOLDOUT_MONTHS,
)
v75_eval.to_csv( OUTPUT_DIR/'v7_5_holdout_evaluation.csv',   index=False)
v75_preds.to_csv(OUTPUT_DIR/'v7_5_holdout_predictions.csv',  index=False)

v75_decomp = {}
for horizon in [3, 1]:
    if horizon not in v75_cal_fc: continue
    v75_decomp[horizon] = build_v75_error_decomposition(
        actuals_df=actuals_df, dim_product_df=dim_prod_df,
        scode_fc=v75_cal_fc[horizon],
        scol_fc=v75_scol_fc[horizon],
        sku_fc=v75_sku_fc[horizon],
        holdout_months=HOLDOUT_MONTHS,
    )

decomp_all = pd.concat([df for df in v75_decomp.values() if not df.empty], ignore_index=True)
decomp_all.to_csv(OUTPUT_DIR/'v7_5_error_decomposition.csv', index=False)

print('=== v7.5 HOLDOUT EVALUATION — Jan–April 2026 ===')
print(v75_eval.to_string(index=False))
print()
print('=== ERROR DECOMPOSITION ===')
print(decomp_all.sort_values(['HorizonMonths','MonthStart','Level']).to_string(index=False))

=== v7.5 HOLDOUT EVALUATION — Jan–April 2026 ===
Level  HorizonMonths MonthStart Segment  N_SKUs  TotalActual  TotalPredicted  AbsError   WMAPE
  SKU              1    2026-01     ALL    1976       949368       981516.55 548791.00 57.8059
  SKU              1    2026-02     ALL    2031       833964      1056453.67 542902.28 65.0990
  SKU              1    2026-03     ALL    2092       950719      1103841.20 714215.96 75.1238
  SKU              1    2026-04     ALL    2034       808962      1051161.19 578120.78 71.4645
  SKU              3    2026-01     ALL    1976       949368       956408.72 524238.76 55.2198
  SKU              3    2026-02     ALL    2031       833964       984748.74 503770.60 60.4068
  SKU              3    2026-03     ALL    2092       950719      1051445.03 676423.18 71.1486
  SKU              3    2026-04     ALL    2034       808962      1023123.43 571163.60 70.6045

=== ERROR DECOMPOSITION ===
     Level  HorizonMonths MonthStart  TotalActual  TotalPredicted  

## Section 11 — Bias Analysis: Raw vs Calibrated

In [16]:
from lane7_forecast.backtest_calibration_v75 import build_bias_analysis

if 3 in v75_raw_fc and 3 in v75_cal_fc:
    bias_analysis = build_bias_analysis(
        actuals_df=actuals_df,
        dim_product_df=dim_prod_df,
        raw_scode_fc=v75_raw_fc[3],
        calibrated_scode_fc=v75_cal_fc[3],
        holdout_months=HOLDOUT_MONTHS,
    )
    bias_analysis.to_csv(OUTPUT_DIR/'v7_5_bias_analysis.csv', index=False)

    print('=== v7.5 BIAS ANALYSIS: Raw vs Calibrated (StyleCode level) ===')
    print(bias_analysis.to_string(index=False))
    print()
    print('Interpretation:')
    print('  BiasImprovement > 0 = calibration moved BiasRatio closer to 1.0')
    print('  CalibratedBiasRatio closer to 1.0 than RawBiasRatio = calibration worked')
    if not bias_analysis.empty:
        overall_raw  = bias_analysis['RawBiasRatio'].mean()
        overall_cal  = bias_analysis['CalibratedBiasRatio'].mean()
        overall_imp  = bias_analysis['BiasImprovement'].mean()
        print(f'\n  Mean RawBiasRatio        : {overall_raw:.4f}')
        print(f'  Mean CalibratedBiasRatio : {overall_cal:.4f}')
        print(f'  Mean BiasImprovement (pp): {overall_imp:+.4f}')

=== v7.5 BIAS ANALYSIS: Raw vs Calibrated (StyleCode level) ===
    Level  HorizonMonths MonthStart  TotalActual  TotalForecastRaw  TotalForecastCalibrated  RawBiasRatio  CalibratedBiasRatio  BiasImprovement
StyleCode              3    2026-01     949368.0         971937.05               1083093.95        1.0238               1.1409           -11.71
StyleCode              3    2026-02     831286.0         997566.49               1110240.69        1.2000               1.3356           -13.56
StyleCode              3    2026-03     950719.0        1064801.49               1183846.80        1.1200               1.2452           -12.52
StyleCode              3    2026-04     808962.0        1052548.39               1169810.05        1.3011               1.4461           -14.50

Interpretation:
  BiasImprovement > 0 = calibration moved BiasRatio closer to 1.0
  CalibratedBiasRatio closer to 1.0 than RawBiasRatio = calibration worked

  Mean RawBiasRatio        : 1.1612
  Mean CalibratedBias

## Section 12 — Version Comparison

In [17]:
from lane7_forecast.production_outputs_v75 import build_v75_version_comparison

_prior_path = next(
    (OUTPUT_DIR/f for f in ['v7_4_vs_prior_versions.csv','v7_3_vs_v7_2_comparison.csv',
                              'v7_2_variant_comparison.csv']
     if (OUTPUT_DIR/f).exists()), None
)

_v74_eval = None
_v74_eval_path = OUTPUT_DIR / 'v7_4_holdout_evaluation.csv'
if _v74_eval_path.exists():
    _v74_eval = pd.read_csv(_v74_eval_path)

comparison = build_v75_version_comparison(
    v75_holdout_eval=v75_eval,
    v75_decomp=decomp_all,
    actuals_df=actuals_df,
    prior_comparison_path=_prior_path,
    v74_holdout_eval=_v74_eval,
    holdout_months=HOLDOUT_MONTHS,
)
comparison.to_csv(OUTPUT_DIR/'v7_5_vs_prior_versions.csv', index=False)

print('=== v7.5 vs PRIOR VERSIONS ===')
print(comparison[[
    'Variant','H1_Jan_WMAPE','H3_Jan_WMAPE','H3_Feb_WMAPE',
    'Overall_H3_WMAPE','BiasRatio','ProductionCandidateFlag'
]].to_string(index=False))
print()
print('Saved: outputs/v7_5_vs_prior_versions.csv')

=== v7.5 vs PRIOR VERSIONS ===
                   Variant  H1_Jan_WMAPE  H3_Jan_WMAPE  H3_Feb_WMAPE  Overall_H3_WMAPE  BiasRatio  ProductionCandidateFlag
           v7.4_production       54.9532       52.4821       55.0256               NaN        NaN                     True
v7.5_calibrated_production       57.8059       55.2198       60.4068           57.8133     1.1584                     True

Saved: outputs/v7_5_vs_prior_versions.csv


## Section 13 — Client Readiness Summary

In [18]:
print("=" * 70)
print("LANE SEVEN DEMAND FORECASTING — v7.5 CLIENT READINESS SUMMARY")
print("=" * 70)
print(f"Evaluation: Detected Holdout Months ({HOLDOUT_MONTHS[0].strftime('%Y-%m')} to {HOLDOUT_MONTHS[-1].strftime('%Y-%m')})")
print(f"Calibration: Backtest-based ({BACKTEST_CONFIG['n_origins']} rolling origins)")
print("Allocation : v7.2 recency_only")
print("=" * 70)
print()

def _get_wmape(df, h, m):
    if df is None or df.empty:
        return float("nan")
    r = df[(df["HorizonMonths"] == h) & (df["MonthStart"] == m)]
    return round(float(r["WMAPE"].iloc[0]), 2) if not r.empty else float("nan")

print("Holdout WMAPE by month:")
for _, row in v75_eval.sort_values(["HorizonMonths", "MonthStart"]).iterrows():
    print(
        f'  H={int(row["HorizonMonths"])} '
        f'{row["MonthStart"]}: {row["WMAPE"]:.1f}%'
    )
print()

tot_act = v75_eval["TotalActual"].sum()
tot_pred = v75_eval["TotalPredicted"].sum()
bias_ratio = round(tot_pred / tot_act, 4) if tot_act > 0 else float("nan")

n_applied = int(calib_table["calibration_applied"].sum()) if not calib_table.empty else 0
n_strong = int((calib_table["evidence_tier"] == "STRONG").sum()) if not calib_table.empty else 0

print(f"Overall BiasRatio     : {bias_ratio:.4f}")
print(f"Calibrations applied  : {n_applied} (STRONG/MODERATE evidence)")
print(f"STRONG evidence       : {n_strong} StyleCode-Horizon pairs")
print()

# Compare to v7.4 if available
if _v74_eval is not None:
    print("Comparison vs v7.4:")
    for _, row in v75_eval.sort_values(["HorizonMonths", "MonthStart"]).iterrows():
        h = int(row["HorizonMonths"])
        m = row["MonthStart"]
        v75_w = row["WMAPE"]
        v74_w = _get_wmape(_v74_eval, h, m)

        if not np.isnan(v74_w):
            print(
                f"  H={h} {m}: v7.5 {v75_w:.1f}% vs v7.4 {v74_w:.1f}% "
                f"({v75_w - v74_w:+.1f}pp)"
            )
print()

# Readiness verdict
green, amber, red = [], [], []

# Use first detected H3 holdout month as primary signal
primary_month = HOLDOUT_MONTHS[0].strftime("%Y-%m")
primary_h3 = _get_wmape(v75_eval, 3, primary_month)

if not np.isnan(primary_h3):
    if primary_h3 < 55:
        green.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% < 55%")
    elif primary_h3 < 70:
        amber.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% in 55–70% range")
    else:
        red.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% > 70%")

if abs(bias_ratio - 1.0) < 0.05:
    green.append(f"Bias near-neutral (ratio={bias_ratio:.3f})")
elif abs(bias_ratio - 1.0) < 0.10:
    amber.append(f"Mild bias (ratio={bias_ratio:.3f})")
else:
    red.append(f"Significant bias (ratio={bias_ratio:.3f})")

if val_report is not None and (~val_report["passed"]).sum() == 0:
    green.append("All validation checks passed")
else:
    red.append("Validation failures present")

for g in green:
    print(f"  ✓ {g}")
for a in amber:
    print(f"  ⚠ {a}")
for r in red:
    print(f"  ✗ {r}")

print()
if not red:
    if not amber:
        print("✅ VERDICT: v7.5 is READY for client presentation.")
    else:
        print("⚠ VERDICT: v7.5 is CONDITIONALLY READY. Review amber signals.")
else:
    print("❌ VERDICT: v7.5 needs further review before client delivery.")

LANE SEVEN DEMAND FORECASTING — v7.5 CLIENT READINESS SUMMARY
Evaluation: Detected Holdout Months (2026-01 to 2026-04)
Calibration: Backtest-based (8 rolling origins)
Allocation : v7.2 recency_only

Holdout WMAPE by month:
  H=1 2026-01: 57.8%
  H=1 2026-02: 65.1%
  H=1 2026-03: 75.1%
  H=1 2026-04: 71.5%
  H=3 2026-01: 55.2%
  H=3 2026-02: 60.4%
  H=3 2026-03: 71.1%
  H=3 2026-04: 70.6%

Overall BiasRatio     : 1.1584
Calibrations applied  : 56 (STRONG/MODERATE evidence)
STRONG evidence       : 52 StyleCode-Horizon pairs

Comparison vs v7.4:
  H=1 2026-01: v7.5 57.8% vs v7.4 55.0% (+2.9pp)
  H=1 2026-02: v7.5 65.1% vs v7.4 61.2% (+3.8pp)
  H=1 2026-03: v7.5 75.1% vs v7.4 70.3% (+4.8pp)
  H=1 2026-04: v7.5 71.5% vs v7.4 66.2% (+5.3pp)
  H=3 2026-01: v7.5 55.2% vs v7.4 52.5% (+2.7pp)
  H=3 2026-02: v7.5 60.4% vs v7.4 55.0% (+5.4pp)
  H=3 2026-03: v7.5 71.1% vs v7.4 64.5% (+6.7pp)
  H=3 2026-04: v7.5 70.6% vs v7.4 62.4% (+8.2pp)

  ✓ All validation checks passed
  ⚠ H=3 2026-01 WMAPE 55.

---
## Run Order

| Cell | Section | Required? |
|------|---------|----------|
| 02 | 0 | Once per env |
| 03 | 0 | Always |
| **04** | **1 Config** | **Always** |
| **05** | **2 Panel + CV** | **Required** |
| **06** | **2 CV** | **Required** |
| **07** | **3 Backtest** | **Required (slow, ~10min)** |
| **08** | **4 Calibration table** | **Required** |
| **09** | **5 Raw forecasts** | **Required** |
| **10** | **6 Apply calibration** | **Required** |
| **11** | **7 Allocation** | **Required** |
| **12** | **8 Production SKU table** | **Required** |
| **13** | **9 Validation** | **Required** |
| **14** | **10 Holdout evaluation** | **Required** |
| **15** | **11 Bias analysis** | **Required** |
| **16** | **12 Version comparison** | **Required** |
| 17 | 13 Summary | Optional |

**Minimum run order:** `03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15 → 16`

> **Section 3 (backtest) is the slowest.** Set `USE_EXISTING_BACKTEST = True` on subsequent runs
> to reload the saved CSV instead of recomputing.